In [12]:
from service.datasetservice.DatasetService import DatasetService

dataset_service = DatasetService()

In [16]:
from scipy.stats import ks_2samp

data_list = list()
for subject in range(2, 36):
    x, _ = dataset_service.get_subject_data(subject=subject, with_features=True)
    _dict = {
        "subject": subject,
        "data": x
    }
    data_list.append(_dict)

for distribution_1 in data_list:
    for distribution_2 in data_list:
        if distribution_1["subject"] != distribution_2["subject"]:
            p_values = ks_2samp(distribution_1["data"]["Deriv_2_RESPR_mean"], distribution_2["data"]["Deriv_2_RESPR_mean"]).pvalue
            if p_values > 0.05:
                print(f"Subject {distribution_1['subject']} and Subject {distribution_2['subject']} have a similar HR: p_value: {p_values}")
            if p_values > 0.05:
                print(f"Subject {distribution_1['subject']} and Subject {distribution_2['subject']} have a similar RR: p_value: {p_values}")

Subject 2 and Subject 10 have a similar HR: p_value: 0.9743547272402832
Subject 2 and Subject 10 have a similar RR: p_value: 0.9743547272402832
Subject 2 and Subject 11 have a similar HR: p_value: 0.5066337203244083
Subject 2 and Subject 11 have a similar RR: p_value: 0.5066337203244083
Subject 2 and Subject 15 have a similar HR: p_value: 0.14366166035385936
Subject 2 and Subject 15 have a similar RR: p_value: 0.14366166035385936
Subject 2 and Subject 16 have a similar HR: p_value: 0.09002325281604628
Subject 2 and Subject 16 have a similar RR: p_value: 0.09002325281604628
Subject 2 and Subject 22 have a similar HR: p_value: 0.20305858162794624
Subject 2 and Subject 22 have a similar RR: p_value: 0.20305858162794624
Subject 2 and Subject 23 have a similar HR: p_value: 0.4671418792332801
Subject 2 and Subject 23 have a similar RR: p_value: 0.4671418792332801
Subject 2 and Subject 30 have a similar HR: p_value: 0.4949541871243409
Subject 2 and Subject 30 have a similar RR: p_value: 0.494

In [11]:
from fastdtw import fastdtw

data_list = list()
for subject in range(2, 36):
    x, _ = dataset_service.get_subject_data(subject=subject, with_features=False)
    _dict = {
        "subject": subject,
        "data": x
    }
    data_list.append(_dict)

results = list()
for distribution_1 in data_list:
    for distribution_2 in data_list:
        if distribution_1["subject"] != distribution_2["subject"]:
            hr_1 = distribution_1["data"]["HR"].to_numpy()
            hr_2 = distribution_2["data"]["HR"].to_numpy()
            rr_1 = distribution_1["data"]["respr"].to_numpy()
            rr_2 = distribution_2["data"]["respr"].to_numpy()
            distance_hr, _ = fastdtw(hr_1, hr_2, dist=2)
            distance_rr, _ = fastdtw(rr_1, rr_2, dist=2)
            _dict = {
                f"{distribution_1['subject']}_{distribution_2['subject']}": {
                    "hr": distance_hr,
                    "rr": distance_rr
                }
            }
            results.append(_dict)

results.sort(key=lambda x: x[list(x.keys())[0]]["hr"])
print(results)

[{'7_24': {'hr': 4559.020000000048, 'rr': 3163.4443664100068}}, {'24_7': {'hr': 4559.020000000048, 'rr': 3163.4443664100068}}, {'21_24': {'hr': 5679.490000000119, 'rr': 1901.4672143099945}}, {'24_21': {'hr': 5679.490000000119, 'rr': 1901.4672143099945}}, {'6_24': {'hr': 5729.680000000019, 'rr': 1380.5960812899978}}, {'24_6': {'hr': 5729.680000000019, 'rr': 1380.5960812899978}}, {'6_27': {'hr': 5817.640000000046, 'rr': 3103.453944220007}}, {'27_6': {'hr': 5817.640000000046, 'rr': 3103.453944220007}}, {'24_27': {'hr': 6138.640000000097, 'rr': 3433.7636424600014}}, {'27_24': {'hr': 6138.640000000097, 'rr': 3433.7636424600014}}, {'4_31': {'hr': 6230.550000000083, 'rr': 2234.242990813}}, {'31_4': {'hr': 6230.550000000083, 'rr': 2234.242990813}}, {'13_24': {'hr': 6261.0100000000975, 'rr': 2942.342884830006}}, {'24_13': {'hr': 6261.0100000000975, 'rr': 2942.342884830006}}, {'21_27': {'hr': 6444.220000000069, 'rr': 2173.866691170007}}, {'27_21': {'hr': 6444.220000000069, 'rr': 2173.86669117000

In [14]:
from collections import defaultdict
from scipy.stats import ks_2samp

data_list = list()
for subject in range(2, 36):
    x, _ = dataset_service.get_subject_data(subject=subject, with_features=True)
    _dict = {
        "subject": subject,
        "data": x
    }
    data_list.append(_dict)

# Dictionaries to count similar features
subject_similarity_count = defaultdict(int)
feature_similarity_count = defaultdict(int)

for distribution_1 in data_list:
    for distribution_2 in data_list:
        if distribution_1["subject"] != distribution_2["subject"]:
            p_values = ks_2samp(distribution_1["data"], distribution_2["data"]).pvalue

            for index, p_value in enumerate(p_values):
                if p_value > 0.05:
                    feature_name = distribution_1["data"].columns[index]
                    subject_similarity_count[distribution_1["subject"]] += 1
                    feature_similarity_count[feature_name] += 1

# Sort and print subjects by the number of similar features
sorted_subjects = sorted(subject_similarity_count.items(), key=lambda item: item[1], reverse=True)
print("Subjects sorted by the number of similar features:")
for subject, count in sorted_subjects:
    print(f"Subject {subject}: {count} similar features")

# Sort and print features by the number of occurrences
sorted_features = sorted(feature_similarity_count.items(), key=lambda item: item[1], reverse=True)
print("Features sorted by the number of occurrences:")
for feature, count in sorted_features:
    print(f"Feature {feature}: {count} occurrences")

Subjects sorted by the number of similar features:
Subject 30: 133 similar features
Subject 15: 118 similar features
Subject 22: 108 similar features
Subject 14: 103 similar features
Subject 23: 93 similar features
Subject 4: 92 similar features
Subject 16: 89 similar features
Subject 28: 89 similar features
Subject 17: 88 similar features
Subject 19: 88 similar features
Subject 25: 85 similar features
Subject 2: 83 similar features
Subject 10: 80 similar features
Subject 18: 79 similar features
Subject 11: 78 similar features
Subject 29: 77 similar features
Subject 3: 76 similar features
Subject 35: 76 similar features
Subject 9: 75 similar features
Subject 20: 74 similar features
Subject 8: 72 similar features
Subject 31: 70 similar features
Subject 32: 69 similar features
Subject 26: 64 similar features
Subject 7: 62 similar features
Subject 5: 61 similar features
Subject 27: 59 similar features
Subject 21: 57 similar features
Subject 33: 57 similar features
Subject 12: 52 similar f